# NIST StRD: univariate summary statistics

Does Julia compute the mean, the standard deviation and the lag-1 autocorrelation to NIST's certified values? The 9 univariate datasets of NIST's Statistical Reference Datasets, from 3 to 5,000 observations, each value printed beside the certified one with the significant digits they share.

The Statistical Reference Datasets (StRD) of the US National Institute of Standards and Technology (NIST) were built to test the numerical accuracy of statistical software. Each comes with values certified to 15 significant digits, computed in multiple precision. Some are real data; others are constructed to be hard, with many leading digits that do not change from one observation to the next, so that software which loses precision gives wrong digits. NIST grades each dataset's difficulty as lower, average or higher.

The data files sit beside this notebook as NIST publishes them (https://www.itl.nist.gov/div898/strd/), and every certified value is read from its file. The number of significant digits a value shares with NIST is the log relative error of McCullough (1998, *The American Statistician* 52(4), 358-366), the usual measure for these datasets: -log10(|value - certified| / |certified|), or -log10|value| when the certified value is 0, rounded down and at most 15.

**The data limit.** Julia stores each number of a file as a Float64, a binary number with 15 to 17 significant digits, and storing alone can change a number: `1000000000000.4` is stored as `1000000000000.4000244140625`. The **data limit** repeats each calculation in 256-bit `BigFloat` arithmetic on the stored numbers: no Float64 program can do better on this data. Where Julia falls short of the data limit, its method lost digits; where the data limit falls short of 15, the storing did.

In [ ]:
Pollis.packages("StatsBase")

## Set-up

`read_strd` reads a NIST file, `certified` reads a certified value from its header, `digits_same` counts the significant digits a value shares with it, `report` prints a dataset's values below NIST's and `summary` the fewest digits in each dataset.

In [ ]:
using Statistics, StatsBase

using Printf

"""
	read_strd(name)

Read the NIST StRD file `name.dat` in this folder.

Gives: `data`, a matrix with one column per variable, the response first, and `header`, the text
above the data, which holds the certified values.
"""
function read_strd(name)
	lines = readlines(name * ".dat")
	range = match(r"Data\s*:?\s*\(?lines\s+(?<first>\d+)\s+to\s+(?<last>\d+)", join(lines[1:12], "\n"))
	first, last = parse(Int, range[:first]), parse(Int, range[:last])
	data = permutedims(reduce(hcat, [parse.(Float64, split(line)) for line in lines[first:last]]))
	return (data = data, header = join(lines[1:first - 1], "\n"))
end

"""
	certified(header, pattern)

The number captured as `v` by `pattern` in a file's header.
"""
certified(header, pattern) = parse(Float64, match(pattern, header)[:v])

"""
	digits_same(value, certified)

The number of significant digits `value` shares with `certified`: McCullough's log relative
error, or the log absolute error when `certified` is 0, rounded down and at most 15.
"""
function digits_same(value, certified)
	isfinite(value) || return 0
	error = certified == 0 ? abs(value) : abs(value - certified) / abs(certified)
	return error == 0 ? 15 : clamp(floor(Int, -log10(error)), 0, 15)
end

"""
	report(title, columns, rows)

Print each certified value with the values of `columns` below it, and the significant digits
each shares with NIST. Each row is `(label, certified, values...)`, one value for each column.

Gives: the fewest digits shared in each column.
"""
function report(title, columns, rows)
	println("\n", title)
	fewest = fill(15, length(columns))
	for (label, c, values...) in rows
		@printf("%-16s%-12s%25.15g\n", label, "certified", c)
		for (k, (column, v)) in enumerate(zip(columns, values))
			d = digits_same(v, c)
			fewest[k] = min(fewest[k], d)
			@printf("%-16s%-12s%25.15g  %s\n", "", column, v, d == 0 ? "DIFFERENT" : "same to $d significant digits")
		end
	end
	return fewest
end

"""
	summary(columns, results)

Print the fewest significant digits shared with NIST in each dataset, one column for each of
`columns`. Each result is `(name, level, fewest)`.
"""
function summary(columns, results)
	@printf("%-12s%-12s", "dataset", "difficulty")
	foreach(column -> @printf("%14s", column), columns)
	println()
	for (name, level, fewest) in results
		@printf("%-12s%-12s", name, level)
		foreach(d -> @printf("%14s", d == 0 ? "DIFFERENT" : string(d)), fewest)
		println()
	end
end

## 1. The datasets

The 9 datasets, with NIST's difficulty. NumAcc1 to NumAcc4 are constructed: 3 values from 10000001 to 10000003, then 1,001 values from 1.1 to 1.3, from 1000000.1 to 1000000.3 and from 10000000.1 to 10000000.3, the same spread behind more and more leading digits that never change.

In [ ]:
datasets = [
	"PiDigits" => "lower", "Lottery" => "lower", "Lew" => "lower", "Mavro" => "lower",
	"Michelso" => "lower", "NumAcc1" => "lower", "NumAcc2" => "average", "NumAcc3" => "average",
	"NumAcc4" => "higher",
]

## 2. Mean, standard deviation, autocorrelation

`mean` and `std` (denominator n - 1) from Statistics, and the lag-1 autocorrelation from StatsBase's `autocor`, which is NIST's definition: the lag-1 autocovariance over the variance, both with denominator n. The data limit is the same three functions on `BigFloat` copies of the stored numbers.

In [ ]:
columns = ["Julia", "data limit"]
results = []
for (name, level) in datasets
	(; data, header) = read_strd(name)
	y = data[:, 1]
	exact = big.(y)
	fewest = report("$name ($level difficulty, $(length(y)) observations)", columns, [
		("mean", certified(header, r"ybar:\s*(?<v>\S+)"), mean(y), mean(exact)),
		("std", certified(header, r"\ss:\s*(?<v>\S+)"), std(y), std(exact)),
		("autocorrelation", certified(header, r"r\(1\):\s*(?<v>\S+)"), autocor(y, [1])[1], autocor(exact, [1])[1]),
	])
	push!(results, (name, level, fewest))
end

## 3. Summary

The fewest significant digits shared with NIST in each dataset.

In [ ]:
summary(columns, results)

## What the summary shows

Julia's fewest digits equal the data limit in 7 of the 9 datasets; they fall short by 2 in PiDigits (its autocorrelation) and by 1 in NumAcc2, the rounding of sums over thousands of values. NumAcc3 and NumAcc4 reach only 9 and 8 digits, but so does the data limit: values such as `10000000.2` cannot be stored exactly in binary, and the error of storing them is all that is lost. `std` computes the variance in two passes, the mean first and then the squared deviations from it, so the leading digits that every value shares cost nothing more.